# Semana 1: Fundamentos Lingüísticos y Primeros Pasos con NLP
## Notebook de Ejercicios (NB2) – Análisis de un Corpus Real

**Propósito:** Aplicar las técnicas de procesamiento de lenguaje natural a un corpus real, el Brown Corpus, para obtener estadísticas, etiquetar partes de la oración, reconocer entidades y reflexionar sobre los desafíos del lenguaje no estructurado.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Cargar y explorar un corpus real usando NLTK.
- Realizar estadísticas básicas (número de oraciones, palabras, vocabulario).
- Aplicar POS tagging y NER con spaCy sobre textos reales.
- Identificar las entidades más frecuentes en el corpus.
- Reflexionar sobre los desafíos del procesamiento de lenguaje no estructurado.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y descargamos los recursos lingüísticos requeridos.

In [ ]:
# Importamos librerías
import nltk
import spacy
from spacy import displacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

# Descargar recursos de NLTK
print("Descargando recursos de NLTK...")
nltk.download('brown', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('tagsets', quiet=True)

# Cargar modelo de spaCy para inglés
try:
    nlp = spacy.load('en_core_web_sm')
    print("Modelo de spaCy cargado correctamente.")
except:
    print("Descargando modelo de spaCy...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

# Configuración de visualización
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga del Brown Corpus

El **Brown Corpus** fue el primer corpus electrónico de inglés americano, compilado en la década de 1960. Contiene aproximadamente 1 millón de palabras de textos publicados en 1961, categorizados en 15 géneros diferentes (noticias, ficción, ciencia, etc.).

In [ ]:
# Cargar el Brown Corpus
from nltk.corpus import brown

print("Brown Corpus cargado.")
print(f"Categorías disponibles: {brown.categories()}")

# Mostrar información básica
print(f"\nNúmero total de palabras: {len(brown.words())}")
print(f"Número total de oraciones: {len(brown.sents())}")

---
## 2. Estadísticas Básicas del Corpus

Realizamos un análisis exploratorio básico del corpus.

In [ ]:
# Obtener todas las palabras y oraciones
words = brown.words()
sents = brown.sents()

# Estadísticas básicas
num_sentences = len(sents)
num_words = len(words)
vocab = set(words)
vocab_size = len(vocab)
avg_sentence_length = num_words / num_sentences

print("=== Estadísticas del Brown Corpus ===")
print(f"Número de oraciones: {num_sentences:,}")
print(f"Número de palabras: {num_words:,}")
print(f"Tamaño del vocabulario: {vocab_size:,}")
print(f"Longitud promedio de oración: {avg_sentence_length:.2f} palabras")
print(f"Densidad léxica (vocabulario/palabras): {vocab_size/num_words:.2%}")

### 2.1. Distribución de la longitud de las oraciones

In [ ]:
# Longitudes de las oraciones
sentence_lengths = [len(sent) for sent in sents]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(sentence_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Longitud de la oración (palabras)')
plt.ylabel('Frecuencia')
plt.title('Distribución de longitud de oraciones')
plt.axvline(avg_sentence_length, color='red', linestyle='--', label=f'Media: {avg_sentence_length:.2f}')
plt.legend()

plt.subplot(1, 2, 2)
plt.boxplot(sentence_lengths)
plt.title('Boxplot de longitud de oraciones')
plt.ylabel('Palabras')

plt.tight_layout()
plt.show()

print(f"Longitud mínima: {min(sentence_lengths)} palabras")
print(f"Longitud máxima: {max(sentence_lengths)} palabras")

### 2.2. Palabras más frecuentes

In [ ]:
# Contar frecuencias de palabras
word_freq = Counter(words)

# Mostrar las 20 palabras más frecuentes
common_words = word_freq.most_common(20)

print("Palabras más frecuentes en el corpus:")
for word, freq in common_words:
    print(f"{word:15} {freq:6,} apariciones")

# Visualizar
words_df = pd.DataFrame(common_words, columns=['Palabra', 'Frecuencia'])
plt.figure(figsize=(12, 6))
plt.bar(words_df['Palabra'], words_df['Frecuencia'])
plt.xlabel('Palabra')
plt.ylabel('Frecuencia')
plt.title('Top 20 palabras más frecuentes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.3. Análisis por categoría (género textual)

El Brown Corpus está dividido en categorías. Analizamos algunas de ellas.

In [ ]:
# Seleccionamos algunas categorías representativas
categorias_interes = ['news', 'fiction', 'science', 'romance']

stats_categorias = []
for cat in categorias_interes:
    cat_words = brown.words(categories=cat)
    cat_sents = brown.sents(categories=cat)
    stats_categorias.append({
        'Categoría': cat,
        'Palabras': len(cat_words),
        'Oraciones': len(cat_sents),
        'Vocabulario': len(set(cat_words)),
        'Longitud promedio': len(cat_words) / len(cat_sents)
    })

df_categorias = pd.DataFrame(stats_categorias)
print("Estadísticas por categoría:")
df_categorias

---
## 3. Aplicación de POS Tagging con spaCy

Debido al tamaño del corpus, trabajaremos con una muestra representativa para POS tagging y NER.

In [ ]:
# Tomamos una muestra de 50 oraciones de la categoría 'news'
news_sents = brown.sents(categories='news')
sample_sents = news_sents[:50]  # primeras 50 oraciones

# Unimos las oraciones en un solo texto
sample_text = ' '.join([' '.join(sent) for sent in sample_sents])

print(f"Muestra creada con {len(sample_sents)} oraciones.")
print(f"Longitud de la muestra: {len(sample_text.split())} palabras.")
print("\nPrimeras 200 caracteres de la muestra:")
print(sample_text[:200])

In [ ]:
# Procesamos la muestra con spaCy
doc = nlp(sample_text)

# Mostramos POS tags para las primeras 20 palabras
print("POS tags para las primeras 20 palabras:")
print(f"{'Palabra':15} {'Tag detallado':15} {'POS simple'}")
print("-" * 50)
for token in list(doc)[:20]:
    print(f"{token.text:15} {token.tag_:15} {token.pos_}")

### 3.1. Distribución de categorías gramaticales

In [ ]:
# Contar frecuencias de POS tags
pos_counts = Counter([token.pos_ for token in doc])

# Mostrar las 10 categorías más comunes
common_pos = pos_counts.most_common(10)

plt.figure(figsize=(10, 6))
pos_df = pd.DataFrame(common_pos, columns=['POS', 'Frecuencia'])
plt.bar(pos_df['POS'], pos_df['Frecuencia'])
plt.xlabel('Categoría gramatical')
plt.ylabel('Frecuencia')
plt.title('Distribución de categorías gramaticales en la muestra')
plt.show()

print("Categorías más frecuentes:")
for pos, freq in common_pos:
    print(f"{pos:10} {freq:6,} apariciones")

---
## 4. Reconocimiento de Entidades Nombradas (NER)

Identificamos entidades como personas, organizaciones, lugares, fechas, etc.

In [ ]:
# Extraer entidades del documento
entities = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]

print(f"Se encontraron {len(entities)} entidades en la muestra.")
print("\nPrimeras 20 entidades encontradas:")
for ent in entities[:20]:
    print(f"- {ent[0]:30} -> {ent[1]:10} ({ent[2]})")

### 4.1. Entidades más frecuentes

In [ ]:
# Contar frecuencia de entidades por texto y por tipo
entity_text_counts = Counter([ent[0] for ent in entities])
entity_type_counts = Counter([ent[1] for ent in entities])

print("Tipos de entidades más frecuentes:")
for ent_type, freq in entity_type_counts.most_common():
    print(f"{ent_type:10} -> {freq} ocurrencias")

print("\nEntidades específicas más frecuentes:")
for ent_text, freq in entity_text_counts.most_common(10):
    print(f"{ent_text:30} -> {freq} ocurrencias")

### 4.2. Visualización de entidades en contexto

Mostramos un fragmento del texto con las entidades resaltadas.

In [ ]:
# Tomamos un fragmento más pequeño para visualizar
fragmento = ' '.join(sample_sents[0])
doc_frag = nlp(fragmento)

# Visualizar entidades
displacy.render(doc_frag, style='ent', jupyter=True)

---
## 5. Reflexión sobre los Desafíos del Lenguaje No Estructurado

El procesamiento de lenguaje natural enfrenta múltiples desafíos debido a la naturaleza del lenguaje humano. Basado en el análisis realizado, reflexionamos sobre algunos de ellos.

### 5.1. Ambigüedad

**Ejemplo encontrado en el corpus:** Buscamos casos de palabras con múltiples significados.

In [ ]:
# Buscar la palabra 'bank' en el corpus
bank_sentences = [sent for sent in sample_sents if any('bank' in word.lower() for word in sent)]

print(f"Oraciones que contienen 'bank':")
for sent in bank_sentences:
    print(f"- {' '.join(sent)}")

### 5.2. Reconocimiento de entidades imperfecto

**Ejemplo de error o ambigüedad en NER:**

In [ ]:
# Buscar posibles errores
print("Analizando posibles errores de NER:")
for ent in entities[:30]:
    if ent[1] == 'PERSON' and len(ent[0].split()) > 3:  # nombres muy largos como persona podrían ser errores
        print(f"Posible error? {ent[0]} -> {ent[1]}")

### 5.3. Variabilidad lingüística

El lenguaje varía según el género textual, la época, el autor, etc.

In [ ]:
# Comparar longitud de oraciones entre categorías
longitudes = []
categorias = ['news', 'fiction', 'science']

for cat in categorias:
    cat_sents = brown.sents(categories=cat)
    cat_lengths = [len(sent) for sent in cat_sents[:100]]  # primeras 100
    longitudes.append(cat_lengths)

plt.figure(figsize=(10, 6))
plt.boxplot(longitudes, labels=categorias)
plt.title('Comparación de longitud de oraciones por categoría')
plt.ylabel('Palabras por oración')
plt.show()

### 5.4. Preguntas para reflexionar

1. **¿Qué tipo de ambigüedades encontraste en el corpus?**
   - Ambigüedad léxica: palabras con múltiples significados (ej. 'bank', 'interest').
   - Ambigüedad sintáctica: frases que pueden interpretarse de varias formas.

2. **¿El etiquetador POS cometió errores? ¿Por qué?**
   - Los errores pueden deberse a palabras poco frecuentes, nombres propios, o contextos ambiguos.

3. **¿El reconocedor de entidades funcionó bien? ¿Qué tipo de entidades detectó correctamente y cuáles no?**
   - Generalmente funciona bien con personas, lugares y organizaciones, pero puede confundir nombres comunes con entidades.

4. **¿Cómo afecta el género textual (noticias vs ficción) a las estadísticas del lenguaje?**
   - Las noticias tienden a tener oraciones más largas y vocabulario más formal; la ficción puede tener más diálogo y variación.

5. **¿Qué desafíos adicionales enfrentaría un sistema NLP al procesar este corpus?**
   - Ruido (errores tipográficos en textos antiguos), variación histórica del lenguaje, jerga específica de dominio.

---
## 6. Conclusiones

En este notebook hemos trabajado con un corpus real (Brown Corpus) y aplicado técnicas básicas de NLP:

✔️ **Estadísticas del corpus**: Calculamos número de oraciones, palabras, vocabulario y distribuciones.

✔️ **POS tagging**: Aplicamos etiquetado gramatical con spaCy y analizamos la distribución de categorías.

✔️ **NER**: Identificamos entidades nombradas y encontramos las más frecuentes.

✔️ **Desafíos del lenguaje**: Reflexionamos sobre ambigüedad, errores de etiquetado y variabilidad lingüística.

**Lección clave**: El lenguaje natural es inherentemente complejo y requiere herramientas sofisticadas para su procesamiento. El análisis exploratorio es fundamental para entender las características del corpus antes de construir modelos.

---
**Fin del Notebook de Ejercicios - Semana 1 (NLP)**